# 골든 셋 v2 근거 대조

골든 셋 v2 100문항(A형 50 + B형 50)이 규칙을 지켰는지 코드로 확인한다.

1. 근거 인용문이 출처 연번의 원문(재해개요·재해유발요인·위험성감소대책·기인물)에 그대로 들어 있는가
2. 기준 답변에 나오는 숫자·연도가 근거 또는 출처 줄에 있는가

읽기만 하고 원본과 골든 셋 파일은 수정하지 않는다.

In [ ]:
import re
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "data" / "processed").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
P = ROOT / "data" / "processed"
G = ROOT / "data" / "golden"

src = {
    "건설업": pd.read_csv(P / "1_SIF_건설업.csv", encoding="utf-8-sig").set_index("연번"),
    "제조업등": pd.read_csv(P / "1_SIF_제조업등.csv", encoding="utf-8-sig").set_index("연번"),
}
norm = lambda s: re.sub(r"[\s▶]+", "", str(s))
COLS = ["재해개요", "재해유발요인", "위험성감소대책(예시)", "기인물", "기인물_표준"]

def pool_of(kind, nums):
    """출처 연번들의 원문을 한 덩어리로 합친다"""
    return "".join(norm(" ".join(str(src[kind].loc[n][c]) for c in COLS))
                   for n in nums if n in src[kind].index)

In [ ]:
rows = []
for f in ["A50", "B50"]:
    text = (G / f"golden_set_v2_{f}_읽기용_final.md").read_text(encoding="utf-8")
    for blk in re.split(r"\n## ", text)[1:]:
        qid = blk.split()[0]
        ans = re.search(r"\*\*기준 답변\*\*\s*\n(.*?)\n\n", blk, re.S).group(1)
        evs = re.findall(r"^- (.*)$", blk.split("**근거 (원문 인용)**")[1].split("**출처**")[0], re.M)
        source = re.search(r"\*\*출처\*\*(.*)", blk).group(1)
        kind = "건설업" if "건설업" in source else "제조업등"
        nums = [int(x) for x in re.findall(r"연번\s*([\d, ]+)", source)[0].replace(" ", "").split(",") if x]
        pool = pool_of(kind, nums)

        # 1) 근거 문장이 원문에 있는가
        bad_ev = []
        for e in evs:
            e = re.sub(r"^\[[^\]]*\]\s*", "", e)
            e = re.sub(r"^(요인|대책|기인물|재해개요)\s*:\s*", "", e)
            if norm(e) not in pool:
                bad_ev.append(e[:40])
        # 2) 기준 답변의 숫자가 근거·출처에 있는가
        miss = [n for n in re.findall(r"\d+(?:\.\d+)?", ans)
                if n not in norm(" ".join(evs) + source)]
        rows.append({"id": qid, "근거 문장 수": len(evs), "원문에 없는 근거": len(bad_ev), "근거에 없는 숫자": len(miss)})

chk = pd.DataFrame(rows)
print("문항 수:", len(chk))
print("근거 문장 수:", chk["근거 문장 수"].sum(), "→ 원문에 없는 근거:", chk["원문에 없는 근거"].sum())
print("기준 답변 숫자가 근거에 없는 문항:", int((chk["근거에 없는 숫자"] > 0).sum()))

한계: 숫자·연도와 근거 문장의 원문 포함 여부만 코드로 확인했다. 기준 답변의 서술이 근거와 의미가 맞는지는 사람이 읽고 확인해야 한다.